# 04_silver_to_gold

Build the Gold reporting layer from the Silver FHIR data.

### Gold responsibilities
- Read the cleaned Silver Delta tables.
- Flatten the FHIR structures into analytics-friendly columns.
- Create reporting-oriented fact and dimension tables in `FHIR_Gold_Warehouse`.
- Create final Warehouse views optimized for reporting and analytics.
- Keep the Gold layer focused on business/reporting use rather than raw FHIR structure.

### Expected Gold objects
- `dim_patient`
- `fact_encounter`
- `fact_observation`
- `fact_condition`
- `vw_patient_summary`
- `vw_encounter_activity`
- `vw_observation_summary`
- `vw_condition_summary`


In [ ]:
from pyspark.sql import functions as F

import com.microsoft.spark.fabric
from com.microsoft.spark.fabric.Constants import Constants

print("Spark version:", spark.version)
print("Fabric Spark connector loaded")

In [2]:
# Lakehouse / Warehouse configuration
# Make FHIR_Silver_Lakehouse the default lakehouse for this notebook.
# Keep FHIR_Gold_Warehouse attached to the notebook as a Warehouse data source.

SILVER_TABLES_PATH = "Tables"
GOLD_WAREHOUSE = "FHIR_Gold_Warehouse"
GOLD_SCHEMA = "dbo"

RESOURCE_TYPES = [
    "patient",
    "encounter",
    "observation",
    "condition"
]

print(f"Silver tables path: {SILVER_TABLES_PATH}")
print(f"Gold warehouse: {GOLD_WAREHOUSE}.{GOLD_SCHEMA}")

StatementMeta(, b28a1571-6dfe-420d-a19b-63bcbbace09c, 4, Finished, Available, Finished, False)

Silver tables path: Tables
Gold warehouse: FHIR_Gold_Warehouse.dbo


## 1. Read Silver Tables

Silver is the source for Gold. Gold should not read the Raw API directly.

In [3]:
def read_silver_resource(resource_type: str):
    table_path = f"{SILVER_TABLES_PATH}/silver_{resource_type}"
    return spark.read.format("delta").load(table_path)


patient_silver_df = read_silver_resource("patient")
patient_silver_df.printSchema()

StatementMeta(, b28a1571-6dfe-420d-a19b-63bcbbace09c, 5, Finished, Available, Finished, False)

root
 |-- patient_id: string (nullable = true)
 |-- source_last_updated: timestamp (nullable = true)
 |-- resource: struct (nullable = true)
 |    |-- _birthDate: struct (nullable = true)
 |    |    |-- extension: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- url: string (nullable = true)
 |    |    |    |    |-- valueDateTime: string (nullable = true)
 |    |-- active: boolean (nullable = true)
 |    |-- address: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- _line: array (nullable = true)
 |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |-- extension: array (nullable = true)
 |    |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |    |-- url: string (nullable = true)
 |    |    |    |    |    |    |    |-- valueString: string (nullable = true)
 |    |    |    |-- city: string (nullable =

## 2. Build Gold Patient Dimension

The Patient dimension exposes commonly used patient attributes while retaining SCD Type 2 columns from Silver.

In [4]:
def build_dim_patient(df):
    return (
        df.select(
            "patient_id",
            F.col("resource.active").alias("active"),
            F.col("resource.gender").alias("gender"),
            F.to_date(F.col("resource.birthDate")).alias("birth_date"),
            F.col("resource.deceasedBoolean").alias("deceased"),
            "source_last_updated",
            "effective_from",
            "effective_to",
            "is_current",
            "source_file"
        )
    )


dim_patient_df = build_dim_patient(patient_silver_df)
display(dim_patient_df.limit(10))

StatementMeta(, b28a1571-6dfe-420d-a19b-63bcbbace09c, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 47ee146d-2cdb-4dd0-91a8-647acf50fdda)

## 3. Build Gold Encounter Fact

Extract the encounter identifier, patient reference, status, class, and period fields needed for reporting.

In [5]:
encounter_silver_df = read_silver_resource("encounter")

fact_encounter_df = encounter_silver_df.select(
    "encounter_id",
    F.regexp_extract(F.col("resource.subject.reference"), r"Patient/(.+)", 1).alias("patient_id"),
    F.col("resource.status").alias("status"),
    F.col("resource.class.code").alias("class_code"),
    F.col("resource.class.display").alias("class_display"),
    F.to_timestamp(F.col("resource.period.start")).alias("period_start"),
    F.to_timestamp(F.col("resource.period.end")).alias("period_end"),
    "source_last_updated",
    "effective_from",
    "effective_to",
    "is_current",
)

display(fact_encounter_df.limit(10))

StatementMeta(, b28a1571-6dfe-420d-a19b-63bcbbace09c, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 65e5c4dc-e3c0-4068-9b65-a3754e5ba90a)

## 4. Build Gold Observation Fact

Extract the observation status, patient reference, code, effective time, and common quantity value/unit fields.

In [6]:
observation_silver_df = read_silver_resource("observation")

fact_observation_df = observation_silver_df.select(
    "observation_id",
    F.regexp_extract(F.col("resource.subject.reference"), r"Patient/(.+)", 1).alias("patient_id"),
    F.col("resource.status").alias("status"),
    F.col("resource.code.coding")[0]["system"].alias("code_system"),
    F.col("resource.code.coding")[0]["code"].alias("code"),
    F.col("resource.code.coding")[0]["display"].alias("code_display"),
    F.col("resource.valueQuantity.value").cast("double").alias("value_quantity"),
    F.col("resource.valueQuantity.unit").alias("value_unit"),
    F.to_timestamp(F.col("resource.effectiveDateTime")).alias("effective_datetime"),
    "source_last_updated",
    "effective_from",
    "effective_to",
    "is_current",
)

display(fact_observation_df.limit(10))

StatementMeta(, b28a1571-6dfe-420d-a19b-63bcbbace09c, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 028df7ac-81f0-4ed9-a78f-e76a4dd98856)

## 5. Build Gold Condition Fact

Extract the condition status, patient reference, clinical/verification status, diagnosis code, and onset information.

In [7]:
condition_silver_df = read_silver_resource("condition")

fact_condition_df = condition_silver_df.select(
    "condition_id",
    F.regexp_extract(F.col("resource.subject.reference"), r"Patient/(.+)", 1).alias("patient_id"),
    F.col("resource.clinicalStatus.coding")[0]["code"].alias("clinical_status"),
    F.col("resource.verificationStatus.coding")[0]["code"].alias("verification_status"),
    F.col("resource.code.coding")[0]["system"].alias("code_system"),
    F.col("resource.code.coding")[0]["code"].alias("code"),
    F.col("resource.code.coding")[0]["display"].alias("code_display"),
    F.to_date(F.col("resource.onsetDateTime")).alias("onset_date"),
    "source_last_updated",
    "effective_from",
    "effective_to",
    "is_current",
)

display(fact_condition_df.limit(10))

StatementMeta(, b28a1571-6dfe-420d-a19b-63bcbbace09c, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ed4da029-1c6d-429f-89cc-ebeefbe6aac2)

## 6. Write Gold Tables to Fabric Warehouse

The Fabric Spark Warehouse connector supports writing Spark DataFrames to Warehouse tables. These tables become the physical Gold reporting layer.

In [12]:
def write_gold_table(df, table_name: str):
    target = f"{GOLD_WAREHOUSE}.{GOLD_SCHEMA}.{table_name}"

    (
        df.write
        .mode("overwrite")
        .synapsesql(target)
    )

    print(f"Written Gold Warehouse table: {target}")


write_gold_table(dim_patient_df, "dim_patient")
write_gold_table(fact_encounter_df, "fact_encounter")
write_gold_table(fact_observation_df, "fact_observation")
write_gold_table(fact_condition_df, "fact_condition")

StatementMeta(, b28a1571-6dfe-420d-a19b-63bcbbace09c, 14, Finished, Available, Finished, False)

Written Gold Warehouse table: FHIR_Gold_Warehouse.dbo.dim_patient
Written Gold Warehouse table: FHIR_Gold_Warehouse.dbo.fact_encounter
Written Gold Warehouse table: FHIR_Gold_Warehouse.dbo.fact_observation
Written Gold Warehouse table: FHIR_Gold_Warehouse.dbo.fact_condition


## 7. Validate Gold Tables

Read the Warehouse tables back through the Fabric Spark Warehouse connector and confirm that records are available.

In [13]:
gold_tables = [
    "dim_patient",
    "fact_encounter",
    "fact_observation",
    "fact_condition"
]

gold_validation = []

for table_name in gold_tables:
    target = f"{GOLD_WAREHOUSE}.{GOLD_SCHEMA}.{table_name}"
    count = spark.read.synapsesql(target).count()
    gold_validation.append(
        (table_name, count, "PASS" if count > 0 else "FAIL")
    )

validation_df = spark.createDataFrame(
    gold_validation,
    ["table_name", "record_count", "status"]
)

display(validation_df)

StatementMeta(, b28a1571-6dfe-420d-a19b-63bcbbace09c, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6009b10d-0d20-46f6-9b94-9372fea2b73f)